# ADAUSDT intraminute features

???????? ?????? ????????? feature dataset ?? ?????? ??????? ???????? `klines` ??? ????????? raw-????.

??? ?????? ?????? ??????????? ??????:

- `open_time`
- `log_close`
- `candle_range`
- `body`
- `upper_wick`
- `lower_wick`
- `body_norm`
- `wick_upper_norm`
- `wick_lower_norm`
- `volume_log`
- `quote_volume_log`
- `taker_buy_ratio`

??? ???????? ?????????????? ?????? ?? ?????? ??????? ??????. ??? ????????????? ??????? ???????????? `0`. ???? ???????? ??????? ???? ???????????? ? ???? parquet-???? ?????????.

In [ ]:
import io
import os
import re

import boto3
import numpy as np
import pandas as pd
from dotenv import load_dotenv


BUCKET = "binance-data-downloader"
RAW_PREFIX = "raw"
FEATURES_PREFIX = "features"
FEATURE_DATASET_NAME = "intraminute_features"
SYMBOL = "ADAUSDT"
INTERVAL = "1m"
SKIP_EXISTING = True

FEATURE_COLUMNS = [
    "open_time",
    "log_close",
    "candle_range",
    "body",
    "upper_wick",
    "lower_wick",
    "body_norm",
    "wick_upper_norm",
    "wick_lower_norm",
    "volume_log",
    "quote_volume_log",
    "taker_buy_ratio",
]


def make_s3_client():
    load_dotenv()
    return boto3.client(
        "s3",
        endpoint_url=os.getenv("YC_ENDPOINT"),
        region_name=os.getenv("YC_REGION"),
        aws_access_key_id=os.getenv("YC_ACCESS_KEY_ID"),
        aws_secret_access_key=os.getenv("YC_SECRET_ACCESS_KEY"),
    )

In [ ]:
def list_symbol_klines_days(
    symbol: str,
    bucket: str = BUCKET,
    raw_prefix: str = RAW_PREFIX,
    interval: str = INTERVAL,
    s3_client=None,
) -> list[str]:
    s3 = s3_client or make_s3_client()
    source_prefix = f"{raw_prefix.strip('/')}/klines/symbol={symbol}/interval={interval}/"
    pattern = re.compile(r"/date=(\d{4}-\d{2}-\d{2})/data\.parquet$")

    dates = set()
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=source_prefix):
        for obj in page.get("Contents", []):
            match = pattern.search(f"/{obj['Key']}")
            if match:
                dates.add(match.group(1))

    return sorted(dates)


def raw_klines_key(
    symbol: str,
    date: str,
    raw_prefix: str = RAW_PREFIX,
    interval: str = INTERVAL,
) -> str:
    return (
        f"{raw_prefix.strip('/')}/klines/"
        f"symbol={symbol}/interval={interval}/date={date}/data.parquet"
    )


def feature_dataset_key(
    symbol: str,
    date: str,
    features_prefix: str = FEATURES_PREFIX,
    feature_dataset_name: str = FEATURE_DATASET_NAME,
    interval: str = INTERVAL,
) -> str:
    return (
        f"{features_prefix.strip('/')}/{feature_dataset_name}/"
        f"symbol={symbol}/interval={interval}/date={date}/data.parquet"
    )


def s3_key_exists(s3_client, bucket: str, key: str) -> bool:
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        return True
    except Exception as exc:
        error_code = getattr(exc, "response", {}).get("Error", {}).get("Code")
        if error_code in {"404", "NoSuchKey", "NotFound"}:
            return False
        raise


def read_symbol_klines_day(
    symbol: str,
    date: str,
    bucket: str = BUCKET,
    raw_prefix: str = RAW_PREFIX,
    interval: str = INTERVAL,
    s3_client=None,
) -> pd.DataFrame:
    s3 = s3_client or make_s3_client()
    key = raw_klines_key(symbol=symbol, date=date, raw_prefix=raw_prefix, interval=interval)
    obj = s3.get_object(Bucket=bucket, Key=key)
    df = pd.read_parquet(io.BytesIO(obj["Body"].read()))

    required_columns = [
        "open_time",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "quote_volume",
        "taker_buy_base",
    ]
    missing_columns = sorted(set(required_columns) - set(df.columns))
    if missing_columns:
        raise ValueError(f"Missing required klines columns: {missing_columns}")

    if df.empty:
        return pd.DataFrame(columns=required_columns)

    cleaned = df[required_columns].copy()
    cleaned["open_time"] = pd.to_numeric(cleaned["open_time"], errors="coerce").astype("Int64")

    numeric_columns = [
        "open",
        "high",
        "low",
        "close",
        "volume",
        "quote_volume",
        "taker_buy_base",
    ]
    for column in numeric_columns:
        cleaned[column] = pd.to_numeric(cleaned[column], errors="coerce")

    cleaned = (
        cleaned
        .dropna(subset=["open_time"])
        .drop_duplicates(subset=["open_time"])
        .sort_values("open_time")
        .reset_index(drop=True)
    )

    open_time_utc = pd.to_datetime(cleaned["open_time"].astype("int64"), unit="ms", utc=True)
    if not open_time_utc.dt.second.eq(0).all() or not open_time_utc.dt.microsecond.eq(0).all():
        raise ValueError("open_time must be minute-aligned UTC")

    return cleaned

In [ ]:
def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    result = np.divide(
        numerator.astype("float64"),
        denominator.astype("float64"),
        out=np.zeros(len(numerator), dtype="float64"),
        where=denominator.astype("float64").to_numpy() != 0,
    )
    return pd.Series(result, index=numerator.index)


def build_intraminute_features_for_day(
    symbol: str,
    date: str,
    bucket: str = BUCKET,
    raw_prefix: str = RAW_PREFIX,
    interval: str = INTERVAL,
    s3_client=None,
) -> pd.DataFrame:
    df = read_symbol_klines_day(
        symbol=symbol,
        date=date,
        bucket=bucket,
        raw_prefix=raw_prefix,
        interval=interval,
        s3_client=s3_client,
    )

    if df.empty:
        return pd.DataFrame(columns=FEATURE_COLUMNS)

    candle_range = df["high"] - df["low"]
    body = df["close"] - df["open"]
    upper_wick = df["high"] - df[["open", "close"]].max(axis=1)
    lower_wick = df[["open", "close"]].min(axis=1) - df["low"]

    feature_df = pd.DataFrame(
        {
            "open_time": df["open_time"].astype("int64"),
            "log_close": np.log(df["close"]),
            "candle_range": candle_range,
            "body": body,
            "upper_wick": upper_wick,
            "lower_wick": lower_wick,
            "body_norm": safe_divide(body, candle_range),
            "wick_upper_norm": safe_divide(upper_wick, candle_range),
            "wick_lower_norm": safe_divide(lower_wick, candle_range),
            "volume_log": np.log1p(df["volume"]),
            "quote_volume_log": np.log1p(df["quote_volume"]),
            "taker_buy_ratio": safe_divide(df["taker_buy_base"], df["volume"]),
        }
    )

    feature_df = feature_df.replace([np.inf, -np.inf], np.nan).fillna(0)
    return feature_df[FEATURE_COLUMNS].reset_index(drop=True)

In [ ]:
def write_intraminute_features_for_symbol_to_s3(
    symbol: str = SYMBOL,
    bucket: str = BUCKET,
    raw_prefix: str = RAW_PREFIX,
    features_prefix: str = FEATURES_PREFIX,
    feature_dataset_name: str = FEATURE_DATASET_NAME,
    interval: str = INTERVAL,
    skip_existing: bool = SKIP_EXISTING,
    s3_client=None,
) -> pd.DataFrame:
    s3 = s3_client or make_s3_client()
    dates = list_symbol_klines_days(
        symbol=symbol,
        bucket=bucket,
        raw_prefix=raw_prefix,
        interval=interval,
        s3_client=s3,
    )

    if not dates:
        raise FileNotFoundError(f"No klines days found for symbol={symbol}, interval={interval}")

    rows = []
    for date in dates:
        key = feature_dataset_key(
            symbol=symbol,
            date=date,
            features_prefix=features_prefix,
            feature_dataset_name=feature_dataset_name,
            interval=interval,
        )

        if skip_existing and s3_key_exists(s3, bucket, key):
            print(f"Skip exists: s3://{bucket}/{key}")
            rows.append({"date": date, "rows": None, "key": key, "status": "skipped"})
            continue

        feature_df = build_intraminute_features_for_day(
            symbol=symbol,
            date=date,
            bucket=bucket,
            raw_prefix=raw_prefix,
            interval=interval,
            s3_client=s3,
        )

        buffer = io.BytesIO()
        feature_df.to_parquet(buffer, index=False, engine="pyarrow", compression="zstd")
        s3.put_object(Bucket=bucket, Key=key, Body=buffer.getvalue())

        print(f"Uploaded: s3://{bucket}/{key} rows={len(feature_df)}")
        rows.append({"date": date, "rows": len(feature_df), "key": key, "status": "uploaded"})

    return pd.DataFrame(rows)

In [ ]:
# ??????? ???????? ?? ????? ??? ????? ???????? ???????.
s3 = make_s3_client()
dates = list_symbol_klines_days(SYMBOL, s3_client=s3)
example_date = dates[0]
example_features = build_intraminute_features_for_day(SYMBOL, example_date, s3_client=s3)

print(example_date, example_features.shape)
display(example_features.head())
display(example_features.tail())
display(example_features.describe(include="all"))

In [ ]:
# ???????? ?????? ???? ??????? parquet-?????? ????????? ? S3.
# ??????????????, ????? ?????? ????? ????????? ?????? ????????.
# write_results = write_intraminute_features_for_symbol_to_s3(s3_client=s3)
# display(write_results)

In [6]:
# ?????? ?? ???? ?????? ?? config.yaml.
from config_loader import load_config

config = load_config("config.yaml")
config_symbols = config["symbols"]
config_interval = config["interval"]
config_start_date = pd.Timestamp(config["date_range"]["start"]).date()
config_end_date = pd.Timestamp(config["date_range"]["end"]).date()
config_bucket = config["storage"]["bucket"]
config_raw_prefix = config["storage"]["prefix"]

s3 = make_s3_client()
all_write_results = []

for symbol in config_symbols:
    available_dates = list_symbol_klines_days(
        symbol=symbol,
        bucket=config_bucket,
        raw_prefix=config_raw_prefix,
        interval=config_interval,
        s3_client=s3,
    )
    selected_dates = [
        date
        for date in available_dates
        if config_start_date <= pd.Timestamp(date).date() <= config_end_date
    ]

    if not selected_dates:
        print(f"No klines days found in config range for symbol={symbol}")
        continue

    rows = []
    for date in selected_dates:
        key = feature_dataset_key(
            symbol=symbol,
            date=date,
            interval=config_interval,
        )

        if SKIP_EXISTING and s3_key_exists(s3, config_bucket, key):
            print(f"Skip exists: s3://{config_bucket}/{key}")
            rows.append({"symbol": symbol, "date": date, "rows": None, "key": key, "status": "skipped"})
            continue

        feature_df = build_intraminute_features_for_day(
            symbol=symbol,
            date=date,
            bucket=config_bucket,
            raw_prefix=config_raw_prefix,
            interval=config_interval,
            s3_client=s3,
        )

        buffer = io.BytesIO()
        feature_df.to_parquet(buffer, index=False, engine="pyarrow", compression="zstd")
        s3.put_object(Bucket=config_bucket, Key=key, Body=buffer.getvalue())

        print(f"Uploaded: s3://{config_bucket}/{key} rows={len(feature_df)}")
        rows.append({"symbol": symbol, "date": date, "rows": len(feature_df), "key": key, "status": "uploaded"})

    all_write_results.append(pd.DataFrame(rows))

write_results_config_period = (
    pd.concat(all_write_results, ignore_index=True)
    if all_write_results
    else pd.DataFrame(columns=["symbol", "date", "rows", "key", "status"])
)

display(write_results_config_period)

Uploaded: s3://binance-data-downloader/features/intraminute_features/symbol=ADAUSDT/interval=1m/date=2020-02-01/data.parquet rows=1439
Uploaded: s3://binance-data-downloader/features/intraminute_features/symbol=ADAUSDT/interval=1m/date=2020-02-02/data.parquet rows=1440
Uploaded: s3://binance-data-downloader/features/intraminute_features/symbol=ADAUSDT/interval=1m/date=2020-02-03/data.parquet rows=1440
Uploaded: s3://binance-data-downloader/features/intraminute_features/symbol=ADAUSDT/interval=1m/date=2020-02-04/data.parquet rows=1440
Uploaded: s3://binance-data-downloader/features/intraminute_features/symbol=ADAUSDT/interval=1m/date=2020-02-05/data.parquet rows=1440
Uploaded: s3://binance-data-downloader/features/intraminute_features/symbol=ADAUSDT/interval=1m/date=2020-02-06/data.parquet rows=1440
Uploaded: s3://binance-data-downloader/features/intraminute_features/symbol=ADAUSDT/interval=1m/date=2020-02-07/data.parquet rows=1440
Uploaded: s3://binance-data-downloader/features/intrami

,symbol,date,rows,key,status
0,ADAUSDT,2020-02-01,1439,features/intraminute_features/symbol=ADAUSDT/i...,uploaded
1,ADAUSDT,2020-02-02,1440,features/intraminute_features/symbol=ADAUSDT/i...,uploaded
2,ADAUSDT,2020-02-03,1440,features/intraminute_features/symbol=ADAUSDT/i...,uploaded
3,ADAUSDT,2020-02-04,1440,features/intraminute_features/symbol=ADAUSDT/i...,uploaded
4,ADAUSDT,2020-02-05,1440,features/intraminute_features/symbol=ADAUSDT/i...,uploaded
...,...,...,...,...,...
2188,ADAUSDT,2026-01-28,1440,features/intraminute_features/symbol=ADAUSDT/i...,uploaded
2189,ADAUSDT,2026-01-29,1440,features/intraminute_features/symbol=ADAUSDT/i...,uploaded
2190,ADAUSDT,2026-01-30,1440,features/intraminute_features/symbol=ADAUSDT/i...,uploaded
2191,ADAUSDT,2026-01-31,1440,features/intraminute_features/symbol=ADAUSDT/i...,uploaded
